# ETL — Associação entre Concentração de Mercado e Preço Real
**TCC — Preços e concentração de mercado em compras públicas de insumos hospitalares, 2009–2023**  
Mônica Anatalia Bezerra de Araujo — MBA em Data Science e Analytics para Operações, POLI USP PRO

**Pergunta:** mercados mais concentrados apresentam precos reais mais altos?

**Por que a correlacao simples nao serve:** correlacionar HHI com o NIVEL de preco
entre itens diferentes mediria o tipo de produto, nao o efeito da concentracao —
um item concentrado pode ser um medicamento caro e um competitivo, uma seringa.
Toda a analise e feita DENTRO do mesmo item, por tres desenhos complementares:

- **A** desvio proporcional do preco em relacao a media do proprio item, por faixa
  de numero de fornecedores
- **B** primeira diferenca: variacao do HHI x variacao do preco, mesmo item, anos
  consecutivos (controla caracteristicas fixas do item)
- **C** correlacao intra-item ao longo do tempo, para itens com serie longa

**Natureza do resultado:** descritivo. Associacao observada nao implica causalidade.

## 1. Setup

In [ ]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
# Ajuste PASTA_DADOS para o local onde estao os arquivos.
# Padrao: subpasta "dados" ao lado do notebook. No Google Colab, aponte para
# a pasta do seu Drive apos monta-lo.
import os
PASTA_DADOS = os.environ.get('BPS_DADOS', 'dados')
# ─────────────────────────────────────────────────────────────────────────
!pip install polars pyarrow scipy --quiet
import polars as pl, numpy as np
from scipy.stats import spearmanr, pearsonr, kruskal
from pathlib import Path

BASE        = Path(PASTA_DADOS)
PASTA_BPS   = BASE / 'Base Harmonizacao Campos'
ARQ_IPCA    = BASE / 'IPCA Tratado' / 'ipca_deflator.parquet'
ARQ_HHI     = BASE / 'Base HHI' / 'hhi_concentracao.parquet'
PASTA_SAIDA = BASE / 'Base Correlacao'
ANO_INICIO, ANO_FIM = 2009, 2023
MIN_ANOS = 5     # item precisa aparecer em ao menos N anos nos desenhos A e C
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

## 2. Montar a base item-ano (preço real mediano + HHI)

In [ ]:
fr=[]
for ano in range(2000, 2026):
    arq = PASTA_BPS / f'BPS_{ano}.parquet'
    if not arq.exists(): continue
    fr.append(pl.read_parquet(arq, columns=['data_compra','preco_unitario','cod_catmat']))
bruto = pl.concat(fr, how='diagonal')
ipca  = pl.read_parquet(ARQ_IPCA).select(['ano','mes','fator_deflacao'])

bps = (bruto.filter(pl.col('data_compra').is_not_null()
                    & pl.col('data_compra').dt.year().is_between(ANO_INICIO, ANO_FIM)
                    & (pl.col('preco_unitario') > 0) & pl.col('cod_catmat').is_not_null())
       .with_columns(pl.col('data_compra').dt.year().alias('ano'),
                     pl.col('data_compra').dt.month().alias('mes'))
       .join(ipca, on=['ano','mes'], how='left')
       .with_columns((pl.col('preco_unitario')*pl.col('fator_deflacao')).alias('preco_real')))
assert bps.filter(pl.col('fator_deflacao').is_null()).height == 0

preco = bps.group_by(['ano','cod_catmat']).agg(pl.col('preco_real').median().alias('preco'))
hhi   = pl.read_parquet(ARQ_HHI).rename({'qtd_fornecedores':'n_forn'})
painel = hhi.join(preco, on=['ano','cod_catmat'], how='inner').filter(pl.col('preco') > 0)
print(f'Pares item-ano com HHI e preco: {painel.height:,}')

longos = painel.group_by('cod_catmat').agg(pl.len().alias('k')).filter(pl.col('k') >= MIN_ANOS)
p5 = painel.join(longos.select('cod_catmat'), on='cod_catmat')
print(f'Restrito a itens com >= {MIN_ANOS} anos: {p5.height:,}')

## 3. Desenho A — desvio proporcional do preço, por número de fornecedores
> Usa-se o desvio em LOG em relacao a media do proprio item. A razao simples
> normalizada pela mediana e degenerada: fica centrada em 1 por construcao e a
> mediana devolve 1,000 em qualquer subgrupo, sem poder de deteccao.

In [ ]:
n = (p5.with_columns(pl.col('preco').log().alias('lp'))
       .with_columns(pl.col('lp').mean().over('cod_catmat').alias('lm'))
       .with_columns((pl.col('lp') - pl.col('lm')).alias('dev')))

n = n.with_columns(pl.when(pl.col('n_forn')==1).then(pl.lit('1 fornecedor'))
      .when(pl.col('n_forn')<=2).then(pl.lit('2 fornecedores'))
      .when(pl.col('n_forn')<=4).then(pl.lit('3 a 4'))
      .when(pl.col('n_forn')<=9).then(pl.lit('5 a 9'))
      .otherwise(pl.lit('10 ou mais')).alias('grupo'))

ORDEM = ['1 fornecedor','2 fornecedores','3 a 4','5 a 9','10 ou mais']
linhas=[]
for g in ORDEM:
    s = n.filter(pl.col('grupo')==g)['dev'].to_numpy()
    linhas.append({'grupo':g,'desvio_medio_pct':round(float(np.mean(s))*100,2),
                   'desvio_mediano_pct':round(float(np.median(s))*100,2),'n':len(s)})
desenho_a = pl.DataFrame(linhas)
print(desenho_a)
H,pv = kruskal(*[n.filter(pl.col('grupo')==g)['dev'].to_numpy() for g in ORDEM])
rs,ps = spearmanr(n['hhi'].to_numpy(), n['dev'].to_numpy())
print(f'\nKruskal-Wallis: H={H:.1f}, p={pv:.2e}')
print(f'Spearman HHI x desvio: {rs:+.4f} (p={ps:.2e})')

## 4. Desenho B — primeira diferença (controla características fixas do item)

In [ ]:
d = (painel.sort(['cod_catmat','ano'])
     .with_columns((pl.col('hhi')-pl.col('hhi').shift(1).over('cod_catmat')).alias('d_hhi'),
                   (pl.col('preco')/pl.col('preco').shift(1).over('cod_catmat')-1).alias('d_preco'),
                   (pl.col('ano')-pl.col('ano').shift(1).over('cod_catmat')).alias('gap'))
     .filter((pl.col('gap')==1) & pl.col('d_hhi').is_not_null() & pl.col('d_preco').is_not_null()))
x, y = d['d_hhi'].to_numpy(), d['d_preco'].to_numpy()
rs_b, ps_b = spearmanr(x, y); rp_b, pp_b = pearsonr(x, y)
print(f'Pares consecutivos: {d.height:,}')
print(f'  Spearman = {rs_b:+.4f} (p={ps_b:.2e})')
print(f'  Pearson  = {rp_b:+.4f} (p={pp_b:.2e})')

q = d.with_columns(pl.col('d_hhi').qcut(4, labels=['Q1 queda forte','Q2 queda','Q3 alta','Q4 alta forte']).alias('quartil'))
desenho_b = (q.group_by('quartil').agg((pl.col('d_preco').median()*100).round(2).alias('var_preco_mediana_pct'),
                                       pl.len().alias('n')).sort('quartil'))
print(desenho_b)

## 5. Desenho C — correlação intra-item ao longo do tempo

In [ ]:
MIN_C = 8
longos_c = painel.group_by('cod_catmat').agg(pl.len().alias('k')).filter(pl.col('k') >= MIN_C)
g = painel.join(longos_c.select('cod_catmat'), on='cod_catmat')
cors=[]
for _, sub in g.group_by('cod_catmat'):
    if sub.height >= MIN_C and sub['hhi'].n_unique() > 1 and sub['preco'].n_unique() > 1:
        r,_ = spearmanr(sub['hhi'].to_numpy(), sub['preco'].to_numpy())
        if not np.isnan(r): cors.append(r)
cors = np.array(cors)
desenho_c = pl.DataFrame({'itens_avaliados':[len(cors)],
                          'correlacao_mediana':[round(float(np.median(cors)),3)],
                          'pct_positivas':[round(float((cors>0).mean())*100,1)],
                          'pct_negativas':[round(float((cors<0).mean())*100,1)]})
print(desenho_c)

## 7. Desenho D — dispersão de preço vs concentração (EXPLORADO E DESCARTADO)
> Registrado por transparencia metodologica. A analise inicial sugeria associacao
> forte (Spearman -0,207) entre concentracao e dispersao de precos. A validacao
> mostrou que o efeito e largamente confundido com o NUMERO DE COMPRAS:
> - estratificando por numero de compras, o coeficiente cai e chega a inverter de sinal
> - fixando a amostra em 10 compras por item-ano, cai de -0,207 para -0,079
> - a exigencia de >=10 compras esvazia as celulas concentradas (44 e 48 observacoes
>   contra 12.440), tornando a comparacao entre extremos insustentavel
>
> **Conclusao: achado descartado.** Mantido no codigo para que a decisao seja auditavel.

In [ ]:
sel = (bps.group_by(['ano','cod_catmat']).agg(pl.len().alias('nc'))
          .filter(pl.col('nc') >= 10).select(['ano','cod_catmat']))
am = (bps.join(sel, on=['ano','cod_catmat'], how='semi')
        .with_columns(pl.int_range(pl.len()).shuffle(seed=7).over(['ano','cod_catmat']).alias('r'))
        .filter(pl.col('r') < 10)
        .group_by(['ano','cod_catmat']).agg(pl.col('preco_real').std().alias('sd'),
                                            pl.col('preco_real').mean().alias('mu'))
        .filter(pl.col('mu') > 0).with_columns((pl.col('sd')/pl.col('mu')).alias('cv10')))
jd = hhi.join(am, on=['ano','cod_catmat'], how='inner')
r_d, p_d = spearmanr(jd['hhi'].to_numpy(), jd['cv10'].to_numpy())
print(f'Com n fixo = 10: Spearman = {r_d:+.4f} (p={p_d:.1e}), n={jd.height:,}')
cel = (jd.with_columns(pl.col('n_forn').cut([1.5,2.5,4.5,9.5],
                        labels=['1','2','3-4','5-9','10+']).alias('g'))
         .group_by('g').agg(pl.col('cv10').median().round(3).alias('cv_mediano'),
                            pl.len().alias('n')).sort('g'))
print(cel)
print('\nCelulas dos mercados concentrados sao residuais -> achado nao sustentavel.')

## 8. Desenho E — comprador x fornecedor: quem explica mais a variação de preço?
> Mede a fracao da variancia do desvio de preco (mesmo item, mesmo ano) explicada
> por cada fator. A comparacao COMPRADOR x FORNECEDOR e justa porque ambos tem
> cardinalidade da mesma ordem; UF e modalidade tem poucas categorias e por isso
> explicam menos por construcao.

In [ ]:
def n14(c):
    return pl.col(c).cast(pl.Utf8).str.replace_all(r'\D','').str.pad_start(14,'0')

fr=[]
for ano in range(ANO_INICIO, ANO_FIM+1):
    arq = PASTA_BPS / f'BPS_{ano}.parquet'
    if not arq.exists(): continue
    fr.append(pl.read_parquet(arq, columns=['data_compra','preco_unitario','cod_catmat',
                                            'uf','cnpj_instituicao','cnpj_fornecedor','modalidade_compra']))
w = (pl.concat(fr, how='diagonal')
     .filter(pl.col('data_compra').is_not_null() & (pl.col('preco_unitario') > 0)
             & pl.col('cod_catmat').is_not_null()
             & pl.col('cnpj_fornecedor').is_not_null() & pl.col('cnpj_instituicao').is_not_null())
     .with_columns(pl.col('data_compra').dt.year().alias('ano'),
                   pl.col('data_compra').dt.month().alias('mes'),
                   n14('cnpj_instituicao').alias('COMPRADOR'),
                   n14('cnpj_fornecedor').alias('FORNECEDOR'))
     .join(ipca, on=['ano','mes'], how='left')
     .with_columns((pl.col('preco_unitario')*pl.col('fator_deflacao')).log().alias('lp'))
     .with_columns(pl.col('lp').median().over(['cod_catmat','ano']).alias('mid'))
     .with_columns((pl.col('lp') - pl.col('mid')).alias('dev')))

tv, mu = w['dev'].var(), w['dev'].mean()
linhas=[]
for c in ['COMPRADOR','FORNECEDOR','uf','modalidade_compra']:
    g = w.group_by(c).agg(pl.col('dev').mean().alias('mm'), pl.len().alias('n'))
    ev = float((((g['mm']-mu)**2)*g['n']).sum()/w.height/tv)
    linhas.append({'fator':c, 'categorias':w[c].n_unique(), 'variancia_explicada_pct':round(ev*100,2)})
desenho_e = pl.DataFrame(linhas).sort('variancia_explicada_pct', descending=True)
print(desenho_e)
razao = (desenho_e.filter(pl.col('fator')=='COMPRADOR')['variancia_explicada_pct'][0]
         / desenho_e.filter(pl.col('fator')=='FORNECEDOR')['variancia_explicada_pct'][0])
nao_exp = 100 - desenho_e['variancia_explicada_pct'].max()
print(f'\nComprador explica {razao:.2f}x o que o fornecedor explica')
print(f'Variacao nao explicada por nenhum fator isolado: ~{nao_exp:.0f}%')

## 6. Exportar

In [ ]:
for nome, df in [('correlacao_desenho_a', desenho_a),
                 ('correlacao_desenho_b', desenho_b),
                 ('correlacao_desenho_c', desenho_c),
                 ('correlacao_desenho_e', desenho_e)]:
    df.write_parquet(PASTA_SAIDA / f'{nome}.parquet')
    df.write_csv(PASTA_SAIDA / f'{nome}.csv')
painel.write_parquet(PASTA_SAIDA / 'painel_item_ano.parquet')
print('=== EXPORTADO ===')
for f in sorted(PASTA_SAIDA.iterdir()): print(' ', f.name)
print('\nValores de referencia para conferencia:')
print('  painel item-ano: 71.739 | itens com >=5 anos: 52.221')
print('  A: 1 fornecedor -4,66% | 10 ou mais -0,49%')
print('  B: 47.952 pares | Spearman -0,0002 (p=0,96)')
print('  C: 3.230 itens | correlacao mediana -0,018 | 48,0% positivas')
print('  D (descartado): Spearman -0,079 com n fixo; celulas de 44 e 48 obs.')
print('  E: COMPRADOR 20,55% | FORNECEDOR 15,50% | UF 2,49% | modalidade 1,74%')

## Memória de cálculo — seção de correlação
> Imprime a conta de cada valor citado no texto do TCC, incluindo o que significa
> cada coeficiente e como se chega as fracoes de variancia explicada.
>
> **Coeficiente de Spearman:** mede se duas variaveis crescem juntas, usando a
> ORDEM dos valores e nao os valores em si — por isso resiste a valores extremos,
> abundantes nesta base. Varia de -1 a +1; zero indica ausencia de relacao.
>
> **valor-p:** probabilidade de observar um coeficiente como esse por acaso, se
> nao houvesse relacao alguma. Acima de 0,05, nao se rejeita a ausencia de relacao.

In [ ]:
print('='*76); print('1. PRIMEIRA DIFERENCA — o desenho principal'); print('='*76)
print('  Para cada item presente em dois anos consecutivos:')
print('    d_hhi   = HHI do ano seguinte  -  HHI do ano anterior')
print('    d_preco = (preco do ano seguinte / preco do ano anterior) - 1')
print('  Compara-se o item consigo mesmo, o que neutraliza suas caracteristicas fixas.')
print(f'\n  pares item-ano consecutivos : {d.height:,}')
print(f'  Spearman                    : {rs_b:+.4f}   (valor-p = {ps_b:.3f})')
print(f'  Pearson                     : {rp_b:+.4f}   (valor-p = {pp_b:.3f})')
print(f'\n  Interpretacao: |{abs(rs_b):.4f}| e praticamente zero e o valor-p de {ps_b:.2f}')
print('  esta muito acima de 0,05 — nao ha evidencia de relacao.')
print('\n  Variacao mediana do preco por quartil de variacao do HHI:')
print('  (quartil = os pares ordenados por d_hhi e divididos em 4 grupos de 25%)')
for r in desenho_b.iter_rows(named=True):
    print(f'    {str(r["quartil"]):<16} {r["var_preco_mediana_pct"]:>+7.2f}%   (n={r["n"]:,})')
print('  Se houvesse relacao, o valor cresceria de Q1 para Q4. Nao cresce.')

In [ ]:
print('='*76); print('2. CORRELACAO INTRA-ITEM AO LONGO DO TEMPO'); print('='*76)
print(f'  Para cada item com pelo menos {MIN_C} anos de registro, calcula-se a correlacao')
print('  entre seu HHI e seu preco ao longo dos anos. Depois resume-se a distribuicao.')
print(desenho_c)
print('\n  Leitura: metade dos itens tem correlacao positiva e metade negativa —')
print('  distribuicao compativel com ausencia de relacao sistematica.')

In [ ]:
print('='*76); print('3. VARIANCIA EXPLICADA — como se calcula'); print('='*76)
print('  Passo 1: para cada registro, dev = log(preco real) - log(mediana do preco')
print('           daquele MESMO item naquele MESMO ano)')
print('  Passo 2: agrupa-se por fator (comprador, fornecedor, UF...) e calcula-se')
print('           a media de dev em cada grupo')
print('  Passo 3: variancia ENTRE grupos dividida pela variancia TOTAL de dev')
print('\n  Quanto maior, mais aquele fator "organiza" a variacao de preco.')
print('  ATENCAO: fatores com mais categorias tendem a explicar mais por construcao.')
print('  Por isso a comparacao valida e COMPRADOR x FORNECEDOR, de cardinalidade')
print('  semelhante; UF e modalidade tem poucas categorias e nao sao comparaveis a eles.\n')
print(f'{"fator":<20} | {"categorias":>10} | {"variancia explicada":>19}')
for r in desenho_e.iter_rows(named=True):
    print(f'{r["fator"]:<20} | {r["categorias"]:>10,} | {r["variancia_explicada_pct"]:>18.2f}%')
comp = desenho_e.filter(pl.col('fator')=='COMPRADOR')['variancia_explicada_pct'][0]
forn = desenho_e.filter(pl.col('fator')=='FORNECEDOR')['variancia_explicada_pct'][0]
print(f'\n  comprador / fornecedor = {comp:.2f} / {forn:.2f} = {comp/forn:.2f}x')
print(f'  nao explicado por nenhum fator isolado: ~{100-comp:.0f}%')
print('\n  Contraste central do trabalho: a IDENTIDADE do fornecedor explica'
      f' {forn:.1f}%,')
print('  enquanto a ESTRUTURA do mercado em que ele atua explica 0,3%.')

In [ ]:
print('='*76); print('4. HIPOTESE DA DISPERSAO — explorada e DESCARTADA'); print('='*76)
print('  Hipotese: mercados concentrados teriam precos mais uniformes entre compradores.')
print(f'  Associacao inicial (sem controle)      : Spearman -0,207')
print(f'  Com amostra fixa em 10 compras por item: Spearman {r_d:+.4f}')
print('\n  Tamanho das celulas apos exigir >= 10 compras:')
for r in cel.iter_rows(named=True):
    print(f'    {str(r["g"]):>5} fornecedores: CV mediano {r["cv_mediano"]:.3f}  (n={r["n"]:,})')
print('\n  MOTIVO DO DESCARTE: a exigencia de >= 10 compras esvazia justamente os')
print('  mercados concentrados. A comparacao entre extremos apoiar-se-ia em algumas')
print('  dezenas de observacoes contra mais de doze mil — amostra insuficiente.')
print('\n  Registrado no codigo para que a decisao seja auditavel.')